# ATML PA3 — Task 3 live generation (Step 4)

Single T4 is enough (Kaggle: Accelerator *GPU T4 x2* or *P100*, Internet on). Attach `atml-pa3-code`, then
*Save & Run All (Commit)*. Resumable: if cut off, attach this notebook's previous output and rerun; finished
calls are restored and skipped. Download `task3_live.zip` from the Output tab.

In [ ]:
# --- locate code (Kaggle dataset "atml-pa3-code" containing atml-pa3-code.zip, or Colab Drive) ---
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content/drive/MyDrive/atml-pa3"
if not ON_KAGGLE:
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
REPO = "/tmp/ATML-PA3"           # code lives outside the saved output
zips = glob.glob("/kaggle/input/**/atml-pa3-code.zip", recursive=True) + glob.glob(f"{WORK}/atml-pa3-code.zip")
# Kaggle usually unpacks an uploaded zip, so also accept an already-extracted copy
dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/task1/train.py", recursive=True)]
shutil.rmtree(REPO, ignore_errors=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(REPO)
else:
    assert dirs, "upload outputs/atml-pa3-code.zip as a Kaggle dataset (or to Drive/atml-pa3/)"
    shutil.copytree(dirs[0], REPO)
print("code from", zips[:1] or dirs[:1])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "hf_xet"], check=True)
os.chdir(REPO)
os.makedirs(f"{WORK}/outputs", exist_ok=True)
if not os.path.exists("outputs"):
    os.symlink(f"{WORK}/outputs", "outputs")
import torch; print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# restore finished calls from an attached earlier version of this notebook
prev = glob.glob("/kaggle/input/**/outputs/task3_live/calls.jsonl", recursive=True) if ON_KAGGLE else []
os.makedirs("outputs/task3_live", exist_ok=True)
if prev and not os.path.exists("outputs/task3_live/calls.jsonl"):
    shutil.copy(prev[0], "outputs/task3_live/calls.jsonl")
    print("restored", sum(1 for _ in open(prev[0])), "calls from", prev[0])
subprocess.run([sys.executable, "-m", "task3.live", "--out", "outputs/task3_live"], check=True)

In [ ]:
n = sum(1 for _ in open("outputs/task3_live/calls.jsonl"))
print(n, "calls (expected 250 = 50 questions x 5 calls)")
shutil.make_archive(f"{WORK}/task3_live", "zip", "outputs/task3_live")
print(f"{WORK}/task3_live.zip", os.path.getsize(f"{WORK}/task3_live.zip") // 1024, "KB")